# 09 — Bag of Words

**Learning objective.** Convert documents into sparse count vectors and inspect exactly what information is preserved or lost.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**tokens → vocabulary counts → sparse document vector → classifier-ready representation**

The key question is not “which API do I call?” but **which representation changes next when I change a control?**

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Switch **counts → binary** | repetition stops increasing feature value | presence matters, frequency does not |
| Change vocabulary | matrix width changes | memory and learned weights change |
| Add n-grams | some word order returns | feature space grows and becomes sparser |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. Why do `dog bites man` and `man bites dog` look identical to unigram BoW?
2. If a word repeats 10 times, how do count and binary vectors differ?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Excellent transparent baseline for short domain text and classification.

### When not to use / caution
Poor choice when meaning depends strongly on order, long context or paraphrase.

### Debugging lens
Inspect the document-term matrix directly; if the representation cannot encode the distinction, no downstream classifier can recover it.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
docs=['ai improves search','search improves discovery','ai search uses representations']
vec=CountVectorizer(binary=False)
X=vec.fit_transform(docs)
frame=pd.DataFrame(X.toarray(),columns=vec.get_feature_names_out(),index=[f'doc{i+1}' for i in range(len(docs))])
frame

      ai  discovery  improves  representations  search  uses
doc1   1          0         1                0       1     0
doc2   0          1         1                0       1     0
doc3   1          0         0                1       1     1

In [3]:
a='dog bites man'; b='man bites dog'
X2=CountVectorizer().fit_transform([a,b])
print('Same unigram BoW vectors?', np.array_equal(X2.toarray()[0],X2.toarray()[1]))
print(X2.toarray())

Same unigram BoW vectors? True
[[1 1 1]
 [1 1 1]]


BoW is a strong baseline because it is cheap, transparent and surprisingly competitive on many classification tasks. Its central limitation is structural: unigram BoW discards order and context.

---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Construct a document-term matrix
- Explain why BoW cannot represent word order